# From Coin Flips to the Bell Curve

**A simulation-based activity on Bernoulli, Binomial, and the emergence of the Gaussian distribution**

---

### Scenario

A website has a signup button. Each visitor either clicks it (a *success*) or doesn't. Suppose the true click rate is 50%. We don't get to see the "true" 50% directly — all we ever observe is a sample of visitors, and the sample result will bounce around a bit each time.

This notebook simulates that process to answer four questions:

1. What does one visitor's outcome look like? (**Bernoulli**)
2. What happens when we count successes across many visitors? (**Binomial**)
3. What happens to our estimate of the click rate as we collect more data? (**Scaling**)
4. Why does a bell-shaped curve (**Gaussian**) show up, no matter what we started with?

No real data, no hypothesis testing — just simulation and observation. Budget about 25–30 minutes.

Run the cells in order. Some cells ask you to write a short prediction *before* running the code below them — do that first, it's the point of the exercise.

## Setup

Run this once. `p` is the *true* click probability we're simulating — in real life you'd never know this, but since we're simulating, we get to set it and check our estimates against it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # fixed seed, so re-running gives the same numbers

p = 0.50   # true click probability, known only because this is a simulation


## 1. One visitor at a time — the Bernoulli variable

A single visitor either clicks (`1`) or doesn't (`0`). That's it — two outcomes, one probability `p` of success. This is called a **Bernoulli random variable**.

**Before running the cell below:** if `p = 0.50`, what do you expect the *average* of 100 such 0/1 outcomes to be close to?

> Your prediction: ______

In [ ]:
visitors = rng.binomial(n=1, p=p, size=100)

print("First 20 outcomes:", visitors[:20])
print("Average of 100 outcomes:", visitors.mean())


**What do you notice?**

- Why can the average of a bunch of 0s and 1s be read as a percentage?
- Change `p = 0.50` to `p = 0.20` in the Setup cell, re-run both cells. What changes, and what stays the same?

## 2. Counting successes — the Binomial variable

Now instead of looking at one visitor, look at a batch of `n` visitors and count how many clicked. That count is a **Binomial random variable** — literally just "add up a bunch of Bernoullis."

**Before running the cell below:** if `p = 0.50` and `n = 100`, what number of successes would you expect *on average*, if you repeated the 100-visitor experiment thousands of times?

> Your prediction: ______

In [ ]:
n = 100
n_experiments = 10_000

# each entry = number of successes in one 100-visitor experiment
counts = rng.binomial(n=n, p=p, size=n_experiments)

print("Average number of successes across", n_experiments, "experiments:", counts.mean())
print("Expected value (n * p):", n * p)

plt.figure(figsize=(7, 4))
plt.hist(counts, bins=range(counts.min(), counts.max() + 2), align='left')
plt.axvline(n * p, color='black', linestyle='--', label='expected value (n·p)')
plt.xlabel("Number of successes out of 100 visitors")
plt.ylabel("Number of experiments with this outcome")
plt.title("Distribution of successes across 10,000 repeated experiments")
plt.legend()
plt.show()


**What do you notice?**

- Does every single experiment give exactly 50 successes? If not, what does?
- Already, with just `n = 100`, does the shape of this histogram remind you of anything?

## 3. From counts to rates — and what happens when you scale up

A raw count of successes is less useful than a *rate*. Define the sample proportion:

```
p_hat = (number of successes) / n
```

`p_hat` is our estimate of the true click rate `p`, built from a sample. Every time we run the experiment, `p_hat` comes out a little different — that's sampling variation.

**Before running the cell below:** as `n` gets bigger (more visitors per experiment), do you expect `p_hat` to bounce around *more* or *less* from one experiment to the next?

> Your prediction: ______

In [ ]:
for n in [10, 100, 1000, 10_000]:
    counts = rng.binomial(n=n, p=p, size=n_experiments)
    p_hat = counts / n
    print(f"n = {n:>6}   spread (std dev) of p_hat = {p_hat.std():.4f}")


**What do you notice?**

- What happens to the spread of `p_hat` as `n` grows?
- Does this match your prediction? If not, what surprised you?
- Why would a company testing a new button want a *large* sample of visitors, not just 10?

## 4. Watching the bell curve emerge

Now put it all together. For a few different values of `n`, simulate 10,000 experiments, compute `p_hat` each time, and look at its distribution.

**Before running the cell below:** which of these do you expect to look most like a smooth bell curve — `n = 1`, `n = 5`, `n = 30`, or `n = 200`? Why?

> Your prediction: ______

In [ ]:
sample_sizes = [1, 5, 30, 200]

fig, axes = plt.subplots(1, len(sample_sizes), figsize=(16, 4), sharey=False)

for ax, n in zip(axes, sample_sizes):
    counts = rng.binomial(n=n, p=p, size=n_experiments)
    p_hat = counts / n

    ax.hist(p_hat, bins=30, density=True, color='steelblue', edgecolor='white')
    ax.set_title(f"n = {n}")
    ax.set_xlabel("p_hat")

axes[0].set_ylabel("density")
fig.suptitle("Distribution of the sample proportion p_hat as n grows")
plt.tight_layout()
plt.show()


**What do you notice?**

- Describe in your own words how the shape changes from `n = 1` to `n = 200`.
- At `n = 1`, `p_hat` can only take two values (0 or 1). Why does that stop being true as `n` grows?
- This bell-shaped pattern showing up — regardless of the fact that we started with something as simple as a 0/1 coin flip — is (informally) the **Central Limit Theorem** in action. You don't need to prove it here; the point of this notebook is that you've now *seen* it happen.

## Wrap-up

Answer briefly, in your own words:

1. What is the difference between a Bernoulli variable and a Binomial variable?
2. What does `p_hat` estimate, and why is it different every time you run the experiment?
3. What happens to the spread of `p_hat` as sample size increases, and why does that matter in practice?
4. In one or two sentences, describe what you saw happen to the histogram shape across `n = 1, 5, 30, 200`.

---
### Optional: where the numbers come from

You don't need this to complete the activity, but if you're curious why the spread shrinks the way it does:

For a single Bernoulli trial, `Var(Y) = p(1-p)`. Averaging `n` independent trials divides the variance by `n²` relative to the sum, giving:

```
Var(p_hat) = p(1-p) / n
```

So the spread (standard deviation) of `p_hat` shrinks proportionally to `1/√n` — which is exactly the shrinking spread you measured in Section 3.